In [1]:
!pip -q install rank-bm25==0.2.2

In [2]:
from google.colab import drive
# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 1. INSTALL AND IMPORT LIBRARIES

import re
import json
import pickle
import hashlib
import time

import numpy as np
import pandas as pd

from pathlib import Path
from rank_bm25 import BM25Okapi

# Main project directory
PROJECT_DIR = Path("/content/drive/MyDrive/NLP Project")

# Use the SAME metadata as FAISS
EMBED_DIR = PROJECT_DIR / "data/embeddings/bge_small"

# Directory for BM25 indexes
BM25_DIR = PROJECT_DIR / "vector_database/bm25/bge_small"
BM25_DIR.mkdir(parents=True, exist_ok=True)

# Four chunking strategies
STRATEGIES = [
    "fixed_size",
    "sentence_based",
    "recursive",
    "section_aware"
]

# BM25 parameters
K1 = 1.5
B = 0.75

print("BM25 directory:", BM25_DIR)

# Verify input files
for strategy in STRATEGIES:

    metadata_file = (
        EMBED_DIR / strategy / "metadata.jsonl"
    )

    if not metadata_file.exists():
        raise FileNotFoundError(metadata_file)

print("All four metadata files found.")


BM25 directory: /content/drive/MyDrive/NLP Project/vector_database/bm25/bge_small
All four metadata files found.


Define the tokenizer and metadata validation

In [4]:
# 2. TOKENIZATION AND METADATA HELPERS

def tokenize(text):
    """
    Simple lexical tokenizer for BM25.

    - Convert to lowercase
    - Extract letters and numbers
    - Split punctuation and hyphens

    Example:
    'GNN-based Fraud Detection'
    -> ['gnn', 'based', 'fraud', 'detection']
    """

    return re.findall(
        r"[a-z0-9]+",
        str(text).lower()
    )


def load_metadata(strategy):
    """
    Load and validate metadata used by FAISS.
    """

    metadata_file = (
        EMBED_DIR / strategy / "metadata.jsonl"
    )

    df = pd.read_json(metadata_file, lines=True)

    required = {
        "vector_id",
        "chunk_id",
        "article_id",
        "title",
        "text",
        "page_numbers",
        "chunking_strategy"
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{strategy}: missing fields {missing}"
        )

    if df.empty:
        raise ValueError(f"{strategy}: empty corpus")

    # Check that chunk IDs are unique
    assert df["chunk_id"].is_unique

    # Verify row-to-vector correspondence
    assert np.array_equal(
        df["vector_id"].to_numpy(dtype=np.int64),
        np.arange(len(df))
    )

    # Validate chunk text and page references
    assert df["text"].map(
        lambda x: isinstance(x, str) and bool(x.strip())
    ).all()

    assert df["page_numbers"].map(
        lambda x: isinstance(x, list) and len(x) > 0
    ).all()

    assert df["chunking_strategy"].eq(strategy).all()

    return df


def metadata_fingerprint(df):
    """
    Create a checksum from the ordered corpus.

    This detects metadata reordering or changes
    between index creation and retrieval.
    """

    digest = hashlib.sha256()

    for vector_id, chunk_id, article_id, text in zip(
        df["vector_id"],
        df["chunk_id"],
        df["article_id"],
        df["text"]
    ):
        record = (
            f"{int(vector_id)}\t{chunk_id}\t"
            f"{article_id}\t{text}\n"
        )

        digest.update(record.encode("utf-8"))

    return digest.hexdigest()


print("Tokenizer and metadata helpers initialized.")
print("Example:", tokenize("Graph Neural Networks for Fraud Detection"))


Tokenizer and metadata helpers initialized.
Example: ['graph', 'neural', 'networks', 'for', 'fraud', 'detection']


Build and save BM25 indexes for all four strategies

In [5]:
# 3. BUILD BM25 INDEXES FOR ALL STRATEGIES

index_results = []
reference_articles = None

for strategy in STRATEGIES:

    print("\n" + "=" * 60)
    print("BUILDING BM25:", strategy.upper())
    print("=" * 60)

    start_time = time.perf_counter()

    # Load the same passages used by FAISS
    metadata = load_metadata(strategy)

    # Ensure all strategies contain the same articles
    current_articles = set(metadata["article_id"])

    if reference_articles is None:
        reference_articles = current_articles

    elif current_articles != reference_articles:
        raise ValueError(
            "Article coverage differs across strategies."
        )

    # Tokenize every passage
    tokenized_corpus = [
        tokenize(text)
        for text in metadata["text"]
    ]

    # Check that tokenization produced usable passages
    if any(len(tokens) == 0 for tokens in tokenized_corpus):
        raise ValueError(
            f"{strategy}: passage with no lexical tokens"
        )

    # Create BM25 index
    bm25 = BM25Okapi(
        tokenized_corpus,
        k1=K1,
        b=B
    )

    # Validate corpus size
    assert bm25.corpus_size == len(metadata)

    # Output directory
    output_dir = BM25_DIR / strategy
    output_dir.mkdir(parents=True, exist_ok=True)

    # Fingerprint protects metadata alignment
    fingerprint = metadata_fingerprint(metadata)

    # Save BM25 index
    index_data = {
        "strategy": strategy,
        "bm25": bm25,
        "fingerprint": fingerprint
    }

    with open(
        output_dir / "bm25_index.pkl",
        "wb"
    ) as file:
        pickle.dump(
            index_data,
            file,
            protocol=pickle.HIGHEST_PROTOCOL
        )

    # Save configuration
    config = {
        "strategy": strategy,
        "algorithm": "BM25Okapi",
        "library": "rank-bm25",
        "library_version": "0.2.2",
        "k1": K1,
        "b": B,
        "tokenizer": "lowercase_alphanumeric_v1",
        "total_passages": len(metadata),
        "total_articles": len(current_articles),
        "metadata_fingerprint": fingerprint
    }

    with open(
        output_dir / "config.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(config, file, indent=4)

    elapsed = time.perf_counter() - start_time

    index_results.append({
        "Strategy": strategy,
        "Passages": len(metadata),
        "Articles": len(current_articles),
        "Average Terms": round(bm25.avgdl, 2),
        "Indexing Seconds": round(elapsed, 2),
        "Status": "Saved"
    })

    print("Passages indexed:", bm25.corpus_size)
    print("Unique articles:", len(current_articles))
    print("Average document length:", round(bm25.avgdl, 2))
    print("Indexing time:", round(elapsed, 2), "seconds")

    del bm25, tokenized_corpus, metadata

# Save experiment summary
summary_df = pd.DataFrame(index_results)

summary_df.to_csv(
    BM25_DIR / "bm25_index_summary.csv",
    index=False
)

print("\nBM25 INDEXING COMPLETED")
display(summary_df)



BUILDING BM25: FIXED_SIZE
Passages indexed: 5703
Unique articles: 180
Average document length: 354.72
Indexing time: 6.92 seconds

BUILDING BM25: SENTENCE_BASED
Passages indexed: 5839
Unique articles: 180
Average document length: 333.9
Indexing time: 6.58 seconds

BUILDING BM25: RECURSIVE
Passages indexed: 6106
Unique articles: 180
Average document length: 335.03
Indexing time: 10.47 seconds

BUILDING BM25: SECTION_AWARE
Passages indexed: 7623
Unique articles: 180
Average document length: 252.7
Indexing time: 6.68 seconds

BM25 INDEXING COMPLETED


,Strategy,Passages,Articles,Average Terms,Indexing Seconds,Status
0,fixed_size,5703,180,354.72,6.92,Saved
1,sentence_based,5839,180,333.90,6.58,Saved
2,recursive,6106,180,335.03,10.47,Saved
3,section_aware,7623,180,252.70,6.68,Saved


Implement BM25 retrieval

In [6]:
# 4. LOAD BM25 INDEX AND IMPLEMENT RETRIEVAL

# Cache indexes to avoid reloading for every question
BM25_CACHE = {}


def load_bm25_index(strategy):
    """
    Load a saved BM25 index and its matching metadata.

    Only load pickle files generated by your own project.
    """

    if strategy not in STRATEGIES:
        raise ValueError("Unknown chunking strategy")

    if strategy in BM25_CACHE:
        return BM25_CACHE[strategy]

    index_file = (
        BM25_DIR / strategy / "bm25_index.pkl"
    )

    metadata = load_metadata(strategy)

    # Load trusted local index
    with open(index_file, "rb") as file:
        index_data = pickle.load(file)

    bm25 = index_data["bm25"]

    # Check model and metadata consistency
    assert isinstance(bm25, BM25Okapi)
    assert index_data["strategy"] == strategy
    assert bm25.corpus_size == len(metadata)

    assert (
        index_data["fingerprint"]
        == metadata_fingerprint(metadata)
    ), "BM25 index and metadata do not match."

    BM25_CACHE[strategy] = (bm25, metadata)

    return bm25, metadata


def bm25_search(question, strategy="recursive", top_k=5):
    """
    Retrieve the top-k passages using BM25.

    Returns:
    - rank
    - score
    - vector ID
    - chunk ID
    - article ID
    - title
    - year
    - page numbers
    - passage text
    """

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question cannot be empty")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer")

    bm25, metadata = load_bm25_index(strategy)

    # Apply the same tokenization used for documents
    query_tokens = tokenize(question)

    if not query_tokens:
        return pd.DataFrame()

    # Calculate BM25 scores for all passages
    scores = bm25.get_scores(query_tokens)

    # Keep only passages with positive lexical scores
    valid_indices = np.flatnonzero(
        np.isfinite(scores) & (scores > 0)
    )

    # Return no evidence when there are no matching passages
    if len(valid_indices) == 0:
        return pd.DataFrame()

    # Rank by descending BM25 score
    # Stable sorting makes ties reproducible
    ranked_indices = valid_indices[
        np.argsort(
            -scores[valid_indices],
            kind="stable"
        )[:top_k]
    ]

    results = []

    for rank, idx in enumerate(ranked_indices, start=1):

        chunk = metadata.iloc[int(idx)]

        results.append({
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "vector_id": int(chunk["vector_id"]),
            "chunk_id": chunk["chunk_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "year": chunk.get("year"),
            "page_numbers": chunk["page_numbers"],
            "section": chunk.get("section", ""),
            "text": chunk["text"]
        })

    return pd.DataFrame(results)


print("BM25 retrieval function initialized.")


BM25 retrieval function initialized.


In [7]:
# 5. TEST BM25 RETRIEVAL

question = (
    "How are graph neural networks used "
    "for financial fraud detection?"
)

TOP_K = 5

for strategy in STRATEGIES:

    print("\n" + "=" * 80)
    print("STRATEGY:", strategy.upper())
    print("=" * 80)

    # Warm the cache before timing retrieval
    load_bm25_index(strategy)

    start = time.perf_counter()

    results = bm25_search(
        question=question,
        strategy=strategy,
        top_k=TOP_K
    )

    latency = time.perf_counter() - start

    if results.empty:
        print("No matching passages found.")
        continue

    for _, row in results.iterrows():

        print("\nRank:", row["rank"])
        print("BM25 Score:", row["score"])
        print("Article:", row["article_id"])
        print("Title:", row["title"])
        print("Year:", row["year"])
        print("Pages:", row["page_numbers"])
        print("Section:", row["section"])

        print("\nRetrieved passage:")
        print(row["text"][:500])

        print("-" * 80)

    print(f"Retrieval time: {latency:.4f} seconds")

print("\nBM25 retrieval testing completed.")



STRATEGY: FIXED_SIZE

Rank: 1
BM25 Score: 26.6766
Article: FD_083
Title: Ensemble of Graph Neural Networks for Enhanced Financial Fraud Detection
Year: 2024
Pages: [1]
Section: 

Retrieved passage:
. INTRODUCTION
As financial transactions become more complicated and interconnected, existing fraud detection technologies are reaching their limits in identifying the many patterns and relationships that seems fraudulent [1]. Given this urgent need,
powerful machine learning algorithms are gaining in popularity. These machine learning techniques are designed to
automatically detect fraud tendencies in data by extracting
statistical features from user profiles, behaviours and transaction summari
--------------------------------------------------------------------------------

Rank: 2
BM25 Score: 25.8613
Article: FD_083
Title: Ensemble of Graph Neural Networks for Enhanced Financial Fraud Detection
Year: 2024
Pages: [2]
Section: 

Retrieved passage:
It highlights the limitations of existing 